**imports**

In [158]:
import gymnasium as gym
import torch.nn as nn
import torch.optim as optim
import torch
import torch.nn.functional as F


In [159]:
class ActorCritic(nn.Module): 
    def __init__(self):
        super().__init__()
        self.shared = nn.Linear(4, 256)
        self.actor = nn.Linear(256, 2)
        self.critic = nn.Linear(256, 1)

    def forward(self, x):
        shared=F.relu(self.shared(x))
        actor=F.softmax(self.actor(shared), dim=-1)
        return actor, self.critic(shared)

In [160]:
model = ActorCritic()  
env = gym.make("CartPole-v1")
optimizer=optim.Adam(model.parameters(), lr=3e-4)


In [161]:
def collect_data(env, model):
    steps = 1024
    state, action, reward, value, logprob, done = [], [], [], [], [], []
    obs, info = env.reset()

    for i in range(steps):
        A_tion, v_alue = model(torch.tensor(obs, dtype=torch.float32))
        distribution = torch.distributions.Categorical(A_tion)
        A_tion = distribution.sample()
        log_prob = distribution.log_prob(A_tion)
        state.append(obs)
        action.append(A_tion)
        value.append(v_alue)
        logprob.append(log_prob)
        obs, rew, terminated, truncated, info = env.step(A_tion.item())
        reward.append(rew)
        done.append(terminated or truncated)
        if terminated or truncated:
            obs, info = env.reset()
    return state, action, reward, value, logprob, done

In [162]:
def Gae(Reward, Value, Done, Lamda, gamma):
    GAE = []
    Sum = 0
    for i in reversed(range(len(Reward))):
        next_value = Value[i+1].detach().item() if i < len(Value) - 1 else 0
        current_value = Value[i].detach().item()
        if Done[i]:
            next_value = 0
            Sum = 0
        TD_Error = Reward[i] + gamma * next_value - current_value
        Sum = TD_Error + Lamda * gamma * Sum
        GAE.append(Sum)
    GAE.reverse()
    return GAE

In [163]:
def PPO(State,Action,Reward,Value,Logprob,GAE,Clip,model, optimizer):
    Action=torch.stack(Action)
    Logprob=torch.stack(Logprob).detach()
    Old_values=torch.stack(Value).squeeze().detach()
    GAE,Reward,State=torch.tensor(GAE, dtype=torch.float32), torch.tensor(Reward, dtype=torch.float32), torch.tensor(State, dtype=torch.float32)
    GAE = (GAE - GAE.mean()) / (GAE.std() + 1e-8)
    Returns = GAE + Old_values
    for _ in range(4):
        action_prob, value = model(State)
        distribution = torch.distributions.Categorical(action_prob)
        new_log_prob = distribution.log_prob(Action)
        ration=torch.exp(new_log_prob - Logprob)
        cliped_ratio=torch.clamp(ration, 1-Clip, 1+Clip)
        Lclip=torch.min(ration*GAE, cliped_ratio*GAE).mean()
        Lvalue=F.mse_loss(value.squeeze(), Returns)
        entropy=distribution.entropy().mean()
        Loss=Lclip -0.5* Lvalue +0.05*entropy
        optimizer.zero_grad()
        (-Loss).backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 0.5)
        optimizer.step()

In [164]:
def evaluate(env, model):
    obs, info = env.reset()
    total = 0
    done = False
    while not done:
        action_prob, _ = model(torch.tensor(obs, dtype=torch.float32))
        action = torch.distributions.Categorical(action_prob).sample()
        obs, rew, terminated, truncated, _ = env.step(action.item())
        total += rew
        done = terminated or truncated
    return total

In [167]:
def training(env, model, optimizer):
    episodes=1000
    for i in range(episodes):
        state ,action, reward, value, logprob, done= collect_data(env, model)
        Advantages=Gae(reward, value, done, Lamda=0.95, gamma=0.99)
        PPO(state, action, reward, value, logprob, Advantages, Clip=0.2, model=model, optimizer=optimizer)
        if i % 50 == 0:
            print(f"Episode {i}, Eval Reward: {evaluate(env, model)}")
    
    return sum(reward)


In [168]:
print('start training')
training(env, model, optimizer)


start training
Episode 0, Eval Reward: 12.0
Episode 50, Eval Reward: 105.0
Episode 100, Eval Reward: 100.0
Episode 150, Eval Reward: 218.0
Episode 200, Eval Reward: 236.0
Episode 250, Eval Reward: 404.0
Episode 300, Eval Reward: 500.0
Episode 350, Eval Reward: 500.0
Episode 400, Eval Reward: 348.0
Episode 450, Eval Reward: 334.0
Episode 500, Eval Reward: 500.0
Episode 550, Eval Reward: 500.0
Episode 600, Eval Reward: 500.0
Episode 650, Eval Reward: 217.0
Episode 700, Eval Reward: 500.0
Episode 750, Eval Reward: 395.0
Episode 800, Eval Reward: 500.0
Episode 850, Eval Reward: 500.0
Episode 900, Eval Reward: 500.0
Episode 950, Eval Reward: 500.0


1024.0